# Evaluation

Notebook to launch `evaluation/evaluate.py` on generated images and save quantitative results.

In [ ]:
import sys
import torch

print(f'Python:  {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:     {torch.cuda.get_device_name(0)}')

In [ ]:
!pip install -q \
    torchmetrics==1.3.2 \
    torch-fidelity==0.3.0 \
    clean-fid==0.1.35 \
    lpips==0.1.4 \
    Pillow pyyaml numpy pandas scipy

In [ ]:
from pathlib import Path
import os

REPO_NAME = 'Few-Shot-Personalization-of-a-Diffusion-Model-for-Industrial-Defect-Synthesis'
REPO_URL = f'https://github.com/alecanc/{REPO_NAME}'
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
if WORK == Path('/'):
    WORK = Path('/tmp')
CLONE_TARGET = WORK / REPO_NAME

def is_project(path):
    return (path / 'defect-synthesis' / 'config.yaml').exists()

candidates = [Path.cwd(), *Path.cwd().parents, CLONE_TARGET, WORK / 'defect-synthesis']
candidates = [p for p in candidates if p != Path('/')]
repo = next((p for p in candidates if is_project(p)), None)

if repo is None:
    if CLONE_TARGET.exists() and any(CLONE_TARGET.iterdir()):
        raise FileNotFoundError(f'{CLONE_TARGET} exists but defect-synthesis/config.yaml was not found inside it')
    !git clone {REPO_URL} {CLONE_TARGET}
    repo = CLONE_TARGET

if not is_project(repo):
    raise FileNotFoundError(f'Repository not found correctly: {repo}')

os.chdir(repo)
CONFIG = repo / 'defect-synthesis' / 'config.yaml'
RESULTS = Path('/kaggle/working/results') if Path('/kaggle').exists() else repo / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)

print(f'Repository: {repo}')
print(f'Config:     {CONFIG}')
print(f'Results:    {RESULTS}')


In [ ]:
from pathlib import Path
import base64

EVALUATE_PY_B64 = '...'

evaluate_path = repo / 'evaluation' / 'evaluate.py'
input_candidates = sorted(Path('/kaggle/input').glob('**/evaluate.py')) if Path('/kaggle/input').exists() else []

if input_candidates:
    evaluate_path.parent.mkdir(parents=True, exist_ok=True)
    evaluate_path.write_text(input_candidates[0].read_text())
    print(f'Copied from Kaggle input: {input_candidates[0]}')
elif not evaluate_path.exists():
    evaluate_path.parent.mkdir(parents=True, exist_ok=True)
    evaluate_path.write_text(base64.b64decode(EVALUATE_PY_B64).decode('utf-8'))
    print(f'Created from embedded copy: {evaluate_path}')
else:
    print(f'Using existing: {evaluate_path}')


In [ ]:
import yaml

with open(CONFIG) as f:
    cfg = yaml.safe_load(f)

for key in ['mvtec_root', 'splits_dir', 'generated']:
    path = Path(cfg['paths'][key])
    print(f'{key:<12} {path} | exists={path.exists()}')

In [ ]:
# Parameters
CATEGORIES = ['bottle']          # Use [] for all categories in config.yaml
DEFECT_TYPES = ['broken_large']  # Use [] for all eval defects in the split file
CONDITIONS = ['zero_shot', 'single_stage1', 'single_stage', 'single_stage_full', 'sweep_weight', 'sweep_prompt']  # Evaluated conditions
METRICS = ['fid', 'kid', 'lpips', 'dino']
N_IMAGES = None                 # Example: 8, or None for all available images
SHOTS_TAG = 'baseline'          # Examples: baseline, shots5, shots10, shots20
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT = RESULTS / f'eval_{SHOTS_TAG}.csv'
VISUALIZE = True                # Generate comparison grids and difference heatmaps

print(f'Device: {DEVICE}')
print(f'Output: {OUTPUT}')

In [ ]:
import subprocess
import sys

if not evaluate_path.exists():
    raise FileNotFoundError(f'evaluate.py not found: {evaluate_path}')

cmd = [
    sys.executable, str(evaluate_path),
    '--config', str(CONFIG),
    '--conditions', *CONDITIONS,
    '--metrics', *METRICS,
    '--shots_tag', SHOTS_TAG,
    '--device', DEVICE,
    '--output', str(OUTPUT),
]

for category in CATEGORIES:
    cmd += ['--category', category]
for defect_type in DEFECT_TYPES:
    cmd += ['--defect_type', defect_type]
if N_IMAGES is not None:
    cmd += ['--n_images', str(N_IMAGES)]

print(' '.join(cmd))
if VISUALIZE:
    cmd += ['--visualize']

subprocess.run(cmd, check=True)

In [ ]:
import pandas as pd

df = pd.read_csv(OUTPUT)
display(df)
print(f'Saved CSV:  {OUTPUT}')
print(f'Saved JSON: {OUTPUT.with_suffix(".json")}')